# 🌦️ Bengaluru Weather Explorer
**Question:** What does a decade of Bengaluru weather look like, and how hard is tomorrow's temperature to predict?

In [ ]:
# Choose the data mode: "sample" runs offline on the synthetic fixture,
# "real" downloads/queries the official source the first time and caches it in data/raw/.
MODE = "sample"
OUT = f"outputs/notebook-{MODE}"

import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import load_data, analysis

Path(OUT).mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
CITY, COMPARE = 'bengaluru', 'chennai'
START, END = '2015-01-01', '2024-12-31'  # used in real mode

> **Sample mode** uses invented data with the real schema. Every chart is stamped *SYNTHETIC SAMPLE*. Change `MODE = "real"` above to run on the official data.

## 1 · The request
POWER returns **gridded estimates** for the coordinates you ask for. Record the query.

In [ ]:
lat, lon = load_data.CITIES[CITY]
print(load_data.build_url(lat, lon, START, END))

## 2 · Load and validate
The parser turns the fill value **-999** into `NaN` before anything else.

In [ ]:
df = load_data.load(MODE, CITY, START, END)
print(df.index.min().date(), '->', df.index.max().date(), len(df), 'days')
df.describe().round(2)

## 3 · The trap: what -999 would have done

In [ ]:
raw_json = json.loads(Path(df.attrs['path']).read_text())
naive = pd.DataFrame(raw_json['properties']['parameter'])
print('Lowest T2M if -999 is kept:', naive['T2M'].min(), '°C')
analysis.missing_summary(df)

## 4 · Monthly view
Months with any missing rainfall day are excluded instead of under-counted.

In [ ]:
m = analysis.monthly(df)
display(Image(f"{OUT}/{analysis.plot_monthly_temperature(m, CITY, Path(OUT), MODE == 'sample')}"))
display(Image(f"{OUT}/{analysis.plot_monthly_precip(m, CITY, Path(OUT), MODE == 'sample')}"))

## 5 · Compare two cities with the same method

In [ ]:
other = load_data.load(MODE, COMPARE, START, END)
clims = {CITY: analysis.climatology(df), COMPARE: analysis.climatology(other)}
display(Image(f"{OUT}/{analysis.plot_comparison(clims, Path(OUT), MODE == 'sample')}"))

## 6 · Baselines for tomorrow's temperature
*Persistence* (tomorrow = today) and *climatology* (average for that day of year).

In [ ]:
b = analysis.next_day_baselines(df)
print('persistence MAE', round(b['persistence_mae'], 3), '°C | climatology MAE', round(b['climatology_mae'], 3), '°C')
display(Image(f"{OUT}/{analysis.plot_baselines(b, Path(OUT), MODE == 'sample')}"))

## 7 · Limitations
- One grid cell represents the whole city.
- Recent days can be revised.
- Precipitation is a satellite/model product.

## Next
Beat both baselines, or compare Bengaluru with Chennai on 2015–2024 in real mode.